# Canopy Water Content Exploration Using EMIT Surface Reflectance Data
*June 19, 2025*

This notebook will be used to calculate equivalent water thickness (EWT) or canopy water content (CWC) over and around the [National Ecological Observatory Network (NEON) Soaproot Saddle (SOAP) field site](https://www.neonscience.org/field-sites/soap). The SOAP site is in the Sierra National Forest in California and the EMIT data we use in this notebook is from July 31, 2023. The CWC calculations will be done using the [Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product](https://www.earthdata.nasa.gov/data/catalog/lpcloud-emitl2arfl-001). This notebook will follow along with an existing notebook created by the Land Processes Distributed Active Archive Center (LP DAAC). The existing notebook is called [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html). Throughout the rest of my notebook below, anything in block quotes is from this existing notebook.

EMIT L2A Dataset Citation:

Green, R. (2022). <i>EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001</i> [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/EMIT/EMITL2ARFL.001 Date Accessed: 2025-06-19

From the existing LP DAAC notebook:

> **Background**

> Equivalent Water Thickness (EWT) is the predicted thickness or absorption path length in centimeters (cm) of water that would be required to yield an observed spectra. In the context of vegetation, this is equivalent to canopy water content (CWC) in g/cm^2 because a cm^3 of water has a mass of 1g.

> CWC can be derived from surface reflectance spectra because they provide information about the composition of the target, including water content. Reflectance is the fraction of incoming solar radiation reflected by Earth’s surface. Different materials reflect varying proportions of radiation based upon their chemical composition and physical properties, giving materials their own unique spectral signature or fingerprint. In particular, liquid water causes characteristic absorption features to appear in the near-infrared wavelengths of the solar spectrum, which enables an estimation of its content.

> CWC correlates with vegetation type and health, as well as wildfire risk. The methods used here to calculate CWC are based on the ISOFIT python package. The Beer-Lambert physical model used to calculate CWC is described in Green et al. (2006) and Bohn et al. (2020). It uses wavelength-dependent absorption coefficients of liquid water to determine the absorption path length as a function of absorption feature depth. Of note, this model does not account for multiple scattering effects within the canopy and may result in overestimation of CWC (Bohn et al., 2020).

> More about the [EMIT mission](https://earth.jpl.nasa.gov/emit/) and [EMIT products](https://www.earthdata.nasa.gov/centers/lp-daac).

> **References**

>* Shrestha, Rupesh. 2023. Equivalent water thickness/canopy water content from hyperspectral data. Jupyter Notebook. Oak Ridge National Laboratory Distributed Active Archive Center. https://github.com/rupesh2/ewt_cwc/tree/main
>* Bohn, N., L. Guanter, T. Kuester, R. Preusker, and K. Segl. 2020. Coupled retrieval of the three phases of water from spaceborne imaging spectroscopy measurements. Remote Sensing of Environment 242:111708. https://doi.org/10.1016/j.rse.2020.111708
>* Green, R.O., T.H. Painter, D.A. Roberts, and J. Dozier. 2006. Measuring the expressed abundance of the three phases of water with an imaging spectrometer over melting snow. Water Resources Research 42:W10402. https://doi.org/10.1029/2005WR004509
>* Thompson, D.R., V. Natraj, R.O. Green, M.C. Helmlinger, B.-C. Gao, and M.L. Eastwood. 2018. Optimal estimation for imaging spectrometer atmospheric correction. Remote Sensing of Environment 216:355–373. https://doi.org/10.1016/j.rse.2018.07.003

> **Requirements** - [NASA Earthdata Account](https://urs.earthdata.nasa.gov/home)
> - *No Python setup requirements if connected to the workshop cloud instance!*
> - Local Only Set up Python Environment - See setup_instructions.md in the /setup/ folder to set up a local compatible Python environment
> - Downloaded necessary files. This is done at the end of the 01_Finding_Concurrent_Data notebook.

> **Learning Objectives:**
> - Calculate CWC of a single pixel
> - Calculate CWC of an ROI

> **Tutorial Outline:**

>3.1 Setup
>
> 3.2 Opening EMIT Data
> 
>3.3 Extracting Reflectance of a Pixel
>
>3.4 Calculating CWC
>
>3.4.1 Single Point
>
>3.4.2 DataFrame of Points
>
>3.5 Applying Inversion in Parallel Across an ROI

## **3.1 Setup**

In [2]:
# Import Packages
import os, sys #python module to create and acces file paths
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np #work with multi-dimensional arrays
import xarray as xr #work with labelled multi-dimenstional arrays
from osgeo import gdal #work with raster and vector geospatial data
import rasterio as rio #work with geospatial raster data
import rioxarray as rxr #work with raster arrays
from matplotlib import pyplot as plt #plotting data
import hvplot.xarray #plot multi-dimensional arrays
import hvplot.pandas #plot DataFrames/Series
import pandas as pd #work with DataFrames
import geopandas as gpd #work with geospatial shapefiles
import earthaccess #search for, download, & stream NASA earth data

from modules.emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset
from modules.ewt_calc import calc_ewt #calculate canopy water content fxn
from scipy.optimize import least_squares #nonlinear least-squares

from modules.test_functions import surfrfl_hvplot_image #plot EMIT surface reflectance bands

ModuleNotFoundError: No module named 'emit_tools'

The code in the cell below is copied from my 05_hr_exploring_neon_emit_reflectance_data_soap.ipynb. I copied it over because the cell below this one where we import emit_xarray things gave me an error (ModuleNotFoundError: No module named 'emit_tools') when I tried to run it with the imports above.

**7/1/25 update:** we believe we figured out why the modules.emit_tools and modules.ewt_calc lines above weren't working. We had to move the .py filed with emit_tools and ewt_calc into a "modules" folder in our notebooks/exploratory/initials folders. So I created that modules folder (C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\notebooks\exploratory\hr\modules) with the "__init__.py", "emit_tools.py", "ewt_calc.py", and "jldp_ras_funcs.py" files inside of it, commented out the two cells below, and changed line 21 in the ewt_calc.py file back to "from modules.emit_tools import emit_xarray, ortho_xr" and saved ewt_calc.py. I will now try rerunning this whole notebook with the hopes that the last cell to calculate cwc works.

In [ ]:
# #create nasa_emit_modules folder
# emit_modules_path = '../../../scripts/nasa_emit_modules'

# #add emit_modules_path to sys.path
# if emit_modules_path not in sys.path:
#     sys.path.append(emit_modules_path) 

# from emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset

Before running the cell below, I had to open the ewt_calc.py file in jupyter notebooks and changed line 21 from “from modules.emit_tools import emit_xarray, ortho_xr” TO “from emit_tools import emit_xarray, ortho_xr”. For some reason, the "modules." part of these original lines of code:

from modules.emit_tools import emit_xarray

from modules.ewt_calc import calc_ewt

is not working. I do not have a 'modules' module.

In [ ]:
# from ewt_calc import calc_ewt
# from scipy.optimize import least_squares #nonlinear least-squares

## **3.2 Opening EMIT Data**

> EMIT L2A Reflectance Data are distributed in a non-orthocorrected spatially raw NetCDF4 (.nc) format consisting of the data and its associated metadata. To work with this data, we will use the emit_xarray function from the emit_tools.py module included in the repository.

We downloaded EMIT L2A reflectance data in previous notebooks for this AOP-EMIT project so we can set a file path:

In [ ]:
#define EMIT file path
emit_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl"
           "/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc")

Open the file with emit_xarray function:

In [ ]:
#learn about emit_xarray function
help(emit_xarray)

In [ ]:
#open emit_fp
emit_ds = emit_xarray(
    #filepath
    emit_fp,
    #orthorectify the dataset
    ortho=True
).load()

#check dataset
emit_ds

In [ ]:
# Commented out to save memory/time
#check reflectance data values after np.nan
# emit_ds.reflectance.data

In [ ]:
#set fill values equal to np.nan to improve visualization
emit_ds.reflectance.data[emit_ds.reflectance.data == -9999] = np.nan

In [ ]:
# Commented out to save memory/time
#check reflectance data values after np.nan
# emit_ds.reflectance.data

Plot one wavelength band to preview the granule scene.

In [ ]:
# Commented out to save memory/time
# emit_850_layer_ds = emit_ds.sel(
#     wavelengths=850,
#     #use nearest valid index value
#     method='nearest')
# surfrfl_hvplot_image(
#     emit_850_layer_ds,
#     plottitle=f"{emit_850_layer_ds.wavelengths:.3f} {emit_850_layer_ds.wavelengths.units}")

## **3.2.1 Cropping EMIT Granule to SOAP flightboxes and Burned and Unburned Tile** *updated on 7/6/25*

To make the rest of this code run quicker and to make the CWC calculation less intensive, I am going to crop the EMIT granule to the SOAP flight boxes now. Once this notebook successfully runs the calc_ewt function, I'll crop to my/our tiles of interest either in addition to the flight boxes or insread. Originally, I had this cropping step down in section **3.5**.

In my 05_hr_exploring_neon_emit_reflectance_data_soap notebook, there is code for cropping EMIT data to the SOAP flightboxes. That is where the code below is from.

#### **3.2.1a Crop EMIT Granule to SOAP flightboxes**

In [ ]:
#open a shapefile of the ROI
aop_flightboxes = gpd.read_file("../../../data/shapefiles/aop_flightboxes/AOP_flightboxesAllSites.shp")
soap_polygon = aop_flightboxes[aop_flightboxes.siteID == 'SOAP']
shape = soap_polygon
shape

In [ ]:
#crop emit_ds to SOAP flightboxes
emit_crop_ds = emit_ds.rio.clip(
    #crop to SOAP polygon geometry
    shape.geometry.values,
    #crop to SOAP polygon CRS
    shape.crs,
    #include all pixels touched by polygon
    all_touched=True)

In [ ]:
#export emit_crop_ds and save to filepath we can use in calc_ewt fxn
emit_crop_ds.to_netcdf("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP.nc")

#define filepath
emit_crop_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP.nc")

#check filepath
emit_crop_fp

In [ ]:
#open emit_crop_fp using code from line 37 of ewt_calc.py
#we did this to see if this helps the final fxn run faster
#we also did this to open the cropped dataset in the same way that the calc_ewt fxn does to see if that helps reduce errors.
emit_crop_ds = xr.open_dataset(emit_crop_fp, decode_coords="all")

#check dataset
emit_crop_ds

In [ ]:
#check emit_crop_ds.reflectance after loading in w/ ewt_calc.py code
emit_crop_ds.reflectance

In [ ]:
#check emit_crop_ds.reflectance after loading in w/ ewt_calc.py code.
#wanting to check NaN values and reflectance values in general
#to make sure they're NaN values and not -3000000 from cropping and exporting above
emit_crop_ds.reflectance.plot.hist()

In [ ]:
#view surface reflectance of cropped area for wavelength closest to 850
emit_crop_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# Commented out to save memory/time
# #view emit_ds 850 wavelength after cropping
# emit_crop_850_layer_ds = emit_crop_ds.sel(
#     wavelengths=850,
#     #use nearest valid index value
#     method='nearest')
# surfrfl_hvplot_image(
#     emit_crop_850_layer_ds,
#     plottitle=f"{emit_crop_850_layer_ds.wavelengths:.3f} {emit_crop_850_layer_ds.wavelengths.units}")

#### **3.2.1b Crop EMIT Granule to SOAP burned tile of interest**

Want to make sure our tiles don't have NaN values if possible
probably need to import neonutilities
also need to identify coordinates


The tile boundary shapefile for this burned tile of interest was downloaded and identified in my 09_hr notebook.

In [ ]:
#open burned tile of interest shapefile

#define filepath for burned tiles
burned_tile_shp_fp = ('..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001'
                   '\\neon-aop-products\\2023\\FullSite\\D17'
                   '\\2023_SOAP_7\\Metadata\\DiscreteLidar'
                   '\\TileBoundary\\shps'
                   '\\NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp')

#write burned tile boundary filepath to geodataframe
burned_tile_gdf = gpd.read_file(burned_tile_shp_fp)

#check geodataframe
burned_tile_gdf

In [ ]:
#crop EMIT granule to burned tile of interest
emit_burn_ds = emit_crop_ds.rio.clip(
    #crop to burned tile polygon geometry
    burned_tile_gdf.geometry.values,
    #crop to burned tile polygon CRS
    burned_tile_gdf.crs,
    #include all pixels touched by polygon
    all_touched=True)
#check emit_burn_ds
emit_burn_ds

In [ ]:
#view surface reflectance of burned area for wavelength closest to 850
emit_burn_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
#export emit_burn_ds and save to filepath we can use in calc_ewt fxn
emit_burn_ds.to_netcdf("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")

#define filepath
emit_burn_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")

#check filepath
emit_burn_fp

In [ ]:
#open emit_burn_fp using code from line 37 of ewt_calc.py
#we did this to see if this helps the final fxn run faster
#we also did this to open the cropped dataset in the same way that the calc_ewt fxn does to see if that helps reduce errors.
emit_burn_ds = xr.open_dataset(emit_burn_fp, decode_coords="all")

#check dataset
emit_burn_ds

In [ ]:
#plot burned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    emit_burn_ds.sel(
        wavelengths=850,
        #use nearest valid index value
        method='nearest'),
    plottitle='SOAP Burned Tile Surface Reflectance, 850.1 nm wavelength')

## **3.3 Extracting Reflectance of a Pixel**

>We can mask out the -.01 values used to represent the region of the spectra with strong atmospheric water vapor absorption features.

In [ ]:
emit_crop_ds['reflectance'].data[:,:,emit_crop_ds['good_wavelengths'].data==0] = np.nan
emit_burn_ds['reflectance'].data[:,:,emit_crop_ds['good_wavelengths'].data==0] = np.nan

>Retrieve the spectra from a sample point by providing a latitude and longitude along with a method using the sel function. This will select the pixel closest to the provided coordinates.

I selected a random point within the EMIT granule shown above

In [ ]:
#define the point of interest
point_ds = (
    #dataset
    emit_crop_ds
    .sel(
        #lat and lon of interest
        latitude=37.0244,
        longitude=-119.2471,
        method='nearest'))

#check point
point_ds

Plot point_ds to see spectra:

In [ ]:
#create point_ds_plot
point_ds_plot = (
    point_ds
    .hvplot
    .line(
        x='wavelengths',
        y='reflectance',
        color='black'
    )
    .opts(
        title=(f"Latitude: {point_ds.latitude.values:.3f} Longitude: {point_ds.longitude.values:.3f}")
    )
)

In [ ]:
#check point_ds_plot
point_ds_plot

In [ ]:
#define the point of interest
point_burn_ds = (
    #dataset
    emit_burn_ds
    .sel(
        #lat and lon of interest
        latitude=37.0295,
        longitude=-119.2662,
        method='nearest'))

#check point
point_burn_ds

In [ ]:
#create point_ds_plot
point_burn_ds_plot = (
    point_burn_ds
    .hvplot
    .line(
        x='wavelengths',
        y='reflectance',
        color='black'
    )
    .opts(
        title=(f"Latitude: {point_burn_ds.latitude.values:.3f} Longitude: {point_burn_ds.longitude.values:.3f}")
    )
)
#check plot
point_burn_ds_plot

## **3.4 Calculating CWC**

>As mentioned in the background we use the surface reflectance to estimate CWC. The unique spectral signatures allow identification and quantification based upon the wavelength-dependent absorption coefficients of liquid water. The EMIT mission has applied similar approaches to identify dust source minerals as well as methane point source emissions. The path length of liquid water absorption can be estimated by utilizing a least squares inversion to minimize the residuals between the EMIT reflectance and the Beer-Lambert model (Green et al.,2006), which relates the wavelength-dependent absorption to the path length the photons are traveling through the material. During the inversion, the path lengths are iteratively adjusted to match the modeled spectra to the EMIT reflectance within the water absorption feature region from 850 to 1100 nm.

>First, define a Beer-Lambert Model function that returns the vector of residuals between measured and modeled surface reflectance.

In [ ]:
# https://github.com/isofit/isofit/blob/main/isofit/inversion/inverse_simple.py#L514C1-L532C17
def beer_lambert_model(x, y, wl, alpha_lw):
    """Function, which computes the vector
    of residuals between measured and modeled
    surface reflectance optimizing for path
    length of surface liquid water based on
    the Beer-Lambert attenuation law.

    Args:
        x: state vector (liquid water path length, intercept, slope)
        y: measurement (surface reflectance spectrum)
        wl: instrument wavelengths
        alpha_lw: wavelength dependent absorption coefficients of liquid water

    Returns:
        resid: residual between modeled and measured surface reflectance
    """

    attenuation = np.exp(-x[0] * 1e7 * alpha_lw)
    rho = (x[1] + x[2] * wl) * attenuation
    resid = rho - y

    return resid

> We need some lab measurements of the complex refractive index of liquid water to obtain the wavelength-dependent absorption coefficients. They are calculated by taking four times the product of Pi and the imaginary part of the refractive index, divided by wavelength. The refractive index of liquid water per wavelength is provided by the k_liquid_water_ice.csv in the data folder. We can also preview this data to get a better understanding of the information we are using.

This [k_liquid_water_ice.csv](https://github.com/nasa/VITALS/blob/main/data/k_liquid_water_ice.csv) file can be found in the [data folder in the EMIT VITALS GitHub Repo](https://github.com/nasa/VITALS/tree/main/data).

In [ ]:
#define file path to k_liquid_water_ice.csv file
#this .csv was originally here: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT
#moved it to this filepath: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\notebooks\exploratory\data
#after I got an error from the calc_ewt fxn that it couldn't find the .csv file.
wp_fp = ("../data/k_liquid_water_ice.csv")

#read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

#check k_wi DataFrame
k_wi.head()

In [ ]:
# Commented out to save memory/time
# #set up plot layout
# fig, axs = plt.subplots(
#     2,4,
#     figsize=(15, 6),
#     sharex=True,
#     sharey=True,
#     constrained_layout=True)
# axs = axs.ravel()
# col_n = 0
# for i in range(0, 7):
#     x = k_wi.iloc[:, col_n+i]
#     y = k_wi.iloc[:, col_n+i+1]
#     axs[i].scatter(x, y)
#     axs[i].set_title(y.name)
#     col_n+=1
# fig.supylabel('imaginary parts of refractive index')
# fig.supxlabel('wavelength')
# plt.show()

Next we will define the function below to get the desired data from the csv file.

In [ ]:
# https://github.com/isofit/isofit/blob/dev/isofit/core/common.py#L461C1-L488C26
def get_refractive_index(k_wi, a, b, col_wvl, col_k):
    """Convert refractive index table entries to numpy array.

    Args:
        k_wi:    variable
        a:       start line
        b:       end line
        col_wvl: wavelength column in pandas table
        col_k:   k column in pandas table

    Returns:
        wvl_arr: array of wavelengths
        k_arr:   array of imaginary parts of refractive index
    """

    wvl_ = []
    k_ = []

    for ii in range(a, b):
        wvl = k_wi.at[ii, col_wvl]
        k = k_wi.at[ii, col_k]
        wvl_.append(wvl)
        k_.append(k)

    wvl_arr = np.asarray(wvl_)
    k_arr = np.asarray(k_)

    return wvl_arr, k_arr

>Lastly, to calculate CWC we define a function that uses least squares optimization to minimize the residuals of our Beer-Lambert Model and find a likely path length of liquid water.

In [ ]:
# https://github.com/isofit/isofit/blob/main/isofit/inversion/inverse_simple.py#L443C1-L511C24
def invert_liquid_water(
    rfl_meas: np.array,
    wl: np.array,
    l_shoulder: float = 850,
    r_shoulder: float = 1100,
    lw_init: tuple = (0.02, 0.3, 0.0002),
    lw_bounds: tuple = ([0, 0.5], [0, 1.0], [-0.0004, 0.0004]),
    ewt_detection_limit: float = 0.5,
    return_abs_co: bool = False,
):
    """Given a reflectance estimate, fit a state vector including liquid water path length
    based on a simple Beer-Lambert surface model.

    Args:
        rfl_meas:            surface reflectance spectrum
        wl:                  instrument wavelengths, must be same size as rfl_meas
        l_shoulder:          wavelength of left absorption feature shoulder
        r_shoulder:          wavelength of right absorption feature shoulder
        lw_init:             initial guess for liquid water path length, intercept, and slope
        lw_bounds:           lower and upper bounds for liquid water path length, intercept, and slope
        ewt_detection_limit: upper detection limit for ewt
        return_abs_co:       if True, returns absorption coefficients of liquid water

    Returns:
        solution: estimated liquid water path length, intercept, and slope based on a given surface reflectance
    """
    
    # Ensure least squares is done with float64 datatype (added)
    wl = np.float64(wl)
    
    # params needed for liquid water fitting
    lw_feature_left = np.argmin(abs(l_shoulder - wl))
    lw_feature_right = np.argmin(abs(r_shoulder - wl))
    wl_sel = wl[lw_feature_left : lw_feature_right + 1]

    # adjust upper detection limit for ewt if specified
    if ewt_detection_limit != 0.5:
        lw_bounds[0][1] = ewt_detection_limit

    # load imaginary part of liquid water refractive index and calculate wavelength dependent absorption coefficient
    # __file__ should live at isofit/isofit/inversion/
    
    
    data_dir_path = "../data"
    path_k = os.path.join(data_dir_path,"k_liquid_water_ice.csv")
    
    #isofit_path = os.path.dirname(os.path.dirname(os.path.dirname(__file__)))
    #path_k = os.path.join(isofit_path, "data", "iop", "k_liquid_water_ice.xlsx")

    # k_wi = pd.read_excel(io=path_k, sheet_name="Sheet1", engine="openpyxl")
    # wl_water, k_water = get_refractive_index(
    #     k_wi=k_wi, a=0, b=982, col_wvl="wvl_6", col_k="T = 20°C"
    # )
    k_wi = pd.read_csv(path_k)
    wl_water, k_water = get_refractive_index(
        k_wi=k_wi, a=0, b=982, col_wvl="wvl_6", col_k="T = 20°C"
    )
    kw = np.interp(x=wl_sel, xp=wl_water, fp=k_water)
    abs_co_w = 4 * np.pi * kw / wl_sel

    rfl_meas_sel = rfl_meas[lw_feature_left : lw_feature_right + 1]

    x_opt = least_squares(
        fun=beer_lambert_model,
        x0=lw_init,
        jac="2-point",
        method="trf",
        bounds=(
            np.array([lw_bounds[ii][0] for ii in range(3)]),
            np.array([lw_bounds[ii][1] for ii in range(3)]),
        ),
        max_nfev=15,
        args=(rfl_meas_sel, wl_sel, abs_co_w),
    )

    solution = x_opt.x

    if return_abs_co:
        return solution, abs_co_w
    else:
        return solution

## **3.4.1 Single Point**

>Now that we have all of the pieces, we can estimate the CWC of a single pixel using our invert_liquid_water function. By default there is a detection limit of 0.5, if we are hitting this threshold we can adjust by changing the ewt_detection_limit argument.

In [ ]:
ewt = invert_liquid_water(
    point_ds.reflectance.values,
    point_ds.wavelengths.values)
print(f"EWT for ({point_ds.longitude.values:.3f},{point_ds.latitude.values:.3f}): {ewt[0]:.3f} cm")

In [ ]:
ewt = invert_liquid_water(
    point_burn_ds.reflectance.values,
    point_burn_ds.wavelengths.values)
print(f"EWT for ({point_burn_ds.longitude.values:.3f},{point_burn_ds.latitude.values:.3f}): {ewt[0]:.3f} cm")

## **3.4.2 DataFrame of Points**

>We can also apply this function to the point data we selected in the previous notebook with the interactive plot.

I am going to skip this section because our data is in DataArrays, which are covered in section 3.5, not DataFrames, which is what this section is for.

## **3.5 Applying Inversion in Parallel Across a ROI**

>In the previous notebook, we subset our region of interest and exported the file. Since the CWC calculation is computationally intensive, it can take a while to process large scenes, so it is more efficient to do this spatial subsetting up front. We can use a function included in the ewt_calc.py module to calculate CWC on a cropped image, and create a cloud-optimized GeoTIFF (COG) file containing the results.

I will use the emit_crop_fp and emit_crop_ds variables I already defined above.

In [ ]:
emit_crop_ds

In [ ]:
emit_burn_ds

Now I will calculate CWC for our ROI, the SOAP flightboxes.

In [ ]:
#set output directory
out_dir = "../../../data"

>Use the calc_ewt function to calculate CWC of the cropped image. This function will also create a COG file containing the CWC results. We can also specify the number of CPUs to use manually with a n_cpu argument, or leave it blank to use all but one of the available CPUs. If we set the return_cwc argument to true, the function will also return the COG.

>This will take some time, about 5 minutes, because we’re doing the calculation for roughly 63,000 pixels. Also note that here we provide the ewt_detection_limit to increase it from the default of 0.5 in the function. We do this because there are several regions containing plants that hold significant quantities of water in this scene.

In [ ]:
help(calc_ewt)

In [ ]:
# %%time
# emit_crop_cwc_ds = calc_ewt(
#     #cropped emit dataset
#     emit_crop_fp,
#     out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

The cell above ran successfully, it took about 13 minutes. The .tif file is here: "C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\dataEMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_cwc.tif". I was unable to open the .tif file with the Windows Photo Viewer and instead was able to open it with the code below from the existing [EMTI CWC notebook](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#cwc-calculation-of-an-roi).

In [ ]:
# #view CWC dataset
# emit_crop_cwc_ds

In [ ]:
# #plot CWC of the SOAP site using surfrfl_hvplot_image fxn
# surfrfl_hvplot_image(
#     emit_crop_cwc_ds,
#     plottitle=f"{emit_crop_cwc_ds.cwc.long_name} ({emit_crop_cwc_ds.cwc.units})",
# clabel="Canopy Water Content (g/cm^2)")

#the code commented out below is the original plotting code from
#the existing EMIT CWC notebook.
# ds_cwc.hvplot.image(x='longitude',y='latitude',cmap='viridis',geo=True, tiles='ESRI', frame_width=720,frame_height=405, alpha=0.7, fontscale=2).opts(
#     title=f"{ds_cwc.cwc.long_name} ({ds_cwc.cwc.units})", xlabel='Longitude',ylabel='Latitude')

In [ ]:
%%time
emit_burn_cwc_ds = calc_ewt(
    #burned emit dataset
    emit_burn_fp,
    out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

#view emit_burn-cwc_ds 
emit_burn_cwc_ds

In [ ]:
#plot CWC of the SOAP site using surfrfl_hvplot_image fxn
surfrfl_hvplot_image(
    emit_burn_cwc_ds,
    plottitle=f"{emit_burn_cwc_ds.cwc.long_name} ({emit_burn_cwc_ds.cwc.units})",
clabel="Canopy Water Content (g/cm^2)")

### **Possible for loop for calc_ewt fxn:**

In [ ]:
# emit_cropped_filepaths = [
#     'burned NEON reflectance',
#     'burned EMIT reflectance',
#     'unburned NEON reflectance',
#     'unburned EMIT reflectance'
# ]


# emit_crop_cwc_ds = calc_ewt(
#     #cropped emit dataset
#     emit_crop_fp,
#     out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )